# Notebook 28 — Multi-omics pathway-score fusion

PSF v0.5 is RNA-centric. The v0.6 `omics` layer adds:

- `ATACScorer` — chromatin-accessibility pathway scores.
- `ProteomicsScorer` — protein-abundance pathway scores.
- `MultiOmicsFusion` — weighted fusion of RNA + ATAC + protein.
- `flag_discordant_pathways` — explicit disagreement flag between
  RNA and protein (informative signal, not noise).

Research use only. Not for clinical decision-making.

In [ ]:
import numpy as np
import pandas as pd

from pathway_subtyping.omics import (
    ATACScorer, ProteomicsScorer, MultiOmicsFusion, FusionWeights,
    flag_discordant_pathways,
)

rng = np.random.default_rng(0)
n_per = 50
n_pathways = 8
cluster_profiles = rng.standard_normal((3, n_pathways)) * 0.9
cluster_ids = np.repeat(np.arange(3), n_per)
true = cluster_profiles[cluster_ids]
n = 3 * n_per

path_cols = [f'PATH_{i}' for i in range(n_pathways)]
cell_idx = [f'cell_{i}' for i in range(n)]
rna     = pd.DataFrame(true + rng.normal(0, 1.6, true.shape), columns=path_cols, index=cell_idx)
protein = pd.DataFrame(true + rng.normal(0, 1.6, true.shape), columns=path_cols, index=cell_idx)
labels  = pd.Series(cluster_ids, index=cell_idx, name='cluster')

## 1. Fuse RNA + protein

In [ ]:
result = MultiOmicsFusion().fuse(
    rna=rna, protein=protein,
    weights=FusionWeights(rna=1.0, protein=1.0),
)
result.fused.head()

## 2. 1-NN cell-type accuracy: fused vs RNA-only

The roadmap acceptance target is >= 3% accuracy improvement over
RNA-only on downstream classification.

In [ ]:
def loo_nn_accuracy(features, labels):
    X = features.loc[labels.index].to_numpy(dtype=float)
    y = labels.to_numpy()
    norms = np.linalg.norm(X, axis=1, keepdims=True); norms[norms == 0] = 1.0
    Xn = X / norms
    sim = Xn @ Xn.T; np.fill_diagonal(sim, -np.inf)
    return float((y[sim.argmax(axis=1)] == y).mean())

print(f'RNA-only:  {loo_nn_accuracy(rna, labels):.3f}')
print(f'Fused:     {loo_nn_accuracy(result.fused, labels):.3f}')

## 3. Learn fusion weights automatically

In [ ]:
fusion = MultiOmicsFusion()
learned = fusion.learn_weights(labels=labels, rna=rna, protein=protein, grid_step=0.2)
print(learned.as_dict())

## 4. Flag RNA vs protein discordance

Discordant pathways carry signal about post-transcriptional regulation
— flag them, don't smooth them away.

In [ ]:
# Inject a pathway where RNA and protein disagree (mock post-transcriptional repression)
rna_disc = rna.copy()
protein_disc = protein.copy()
rna_disc['PATH_0'] = rng.standard_normal(n) * 0.5   # low variance
protein_disc['PATH_0'] = 3.0 + rng.standard_normal(n) * 1.5  # shifted + high variance
report = flag_discordant_pathways(rna_disc, protein_disc)
print(report.summary())
print('discordant pathways:', report.discordant_pathways)

## See also
- PSF v0.6 roadmap — Phase 3 F10: [docs/roadmap-v06-codeberg.md](../../docs/roadmap-v06-codeberg.md)